# Deduplicate Data

Remove near-similar tweet

## A. Overview

Apply Reciprocal Rank Fusion (RRF) Hybrid seach using n-gram and all-MiniLM-L6-v2 Transformer model to filter out similar tweet

In [1]:
%pip install datasketch sentence-transformers
# install either faiss-cuda or faiss-cpu depending on your environment
# %pip install faiss-cuda
# %pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


## B. Combine Datasets

In [2]:
from pathlib import Path
import csv
import os
import random
import json
import re

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import login

from datasketch import MinHash, MinHashLSH

import configuration
from src import data_utils, setup
from src.normalizer import similarity

login(os.getenv("HF_TOKEN"))

/opt/homebrew/Caskroom/miniconda/base/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
dataset_path = Path('..') / 'data'

embedding_model="all-MiniLM-L6-v2"
nprobe = 10
chunk_size=50000
top_k = 100

### B.1. Load Datasets

In [4]:
disaster_filepath = dataset_path / 'disaster'
df_disaster = pd.read_csv(disaster_filepath / 'full.csv')
df_disaster['subset'] = 'disaster'
df_disaster.shape

(244403, 5)

In [5]:
# n_disaster = len(df_disaster)
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
nlist_disaster = int(4 * np.sqrt(len(df_disaster)))
embeddings_file_path = disaster_filepath / "disaster_informative_embeddings.npz"

similarity.filter_duplicates_faiss(
    df_disaster,
    nlist=nlist_disaster,
    chunk_size=chunk_size,
    embedding_model="all-MiniLM-L6-v2",
    train_sample_size=100000,
    nprobe=nprobe,
    search_type="radius",
    top_k=top_k,
    similarity_threshold=0.75,
    embeddings_file_path=embeddings_file_path,
    checkpoint_file="disaster_deduplication_checkpoint.json",
)

Original dataset size: 244403
Loading embedding model 'all-MiniLM-L6-v2' on cpu...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16618.45it/s]


Dataset has 244403 rows. Embedding dimension: 384
Pre-allocating embedding array of shape (244403, 384)...
Vectorizing text to dense representations...
Processing rows 0 to 200000 / 244403


Batches:   0%|          | 0/98 [00:42<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# extended_filepath = dataset_path / "extended"

# df_weather = pd.read_csv(
#     extended_filepath / "weather.csv"
# )["tweet_text"].to_frame()
# df_weather["informative"] = False
# df_weather["subset"] = "weather"

# df_out_topic = pd.read_csv(
#     extended_filepath / "out_topic.csv"
# )["tweet_text"].to_frame()
# df_out_topic["informative"] = False
# df_out_topic["subset"] = "out_topic"

## C. Filtering

### C.1. Embedding

In [ ]:
embedding_model="all-MiniLM-L6-v2"
nprobe = 10
chunk_size=50000
similarity_threshold=0.6
top_k = 100

In [ ]:
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
n_disaster = len(df_disaster)
nlist_disaster = int(4 * np.sqrt(n_disaster))
train_sample_size_disaster = 80_000

df_disaster_embeddings = similarity.vectorize_faiss(
    df_disaster, text_column='tweet_text', model_name=embedding_model
)

Loading embedding model 'all-MiniLM-L6-v2' on cuda...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Dataset has 244403 rows. Embedding dimension: 384
Pre-allocating embedding array of shape (244403, 384)...
Vectorizing text to dense representations...
Processing rows 0 to 200000 / 244403


Batches:   0%|          | 0/98 [00:00<?, ?it/s]

Processing rows 200000 to 244403 / 244403


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [30]:
# save the embeddings to disk
np.savez(
    extended_filepath / "df_disaster_informative_embeddings.npz",
    embeddings=df_disaster_informative_embeddings,
    model_name=embedding_model,
)

In [7]:
# # rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
# n_weather = len(df_weather)
# nlist_weather = int(4 * np.sqrt(n_weather))
# train_sample_size_weather = int(n_weather / 10)

# df_weather_embeddings = similarity.vectorize_faiss(
#     df_weather, text_column='tweet_text', model_name=embedding_model
# )

In [8]:
# # save the embeddings to disk
# np.savez(
#     extended_filepath / "df_weather_embeddings.npz",
#     embeddings=df_weather_embeddings,
#     model_name=embedding_model,
# )

### C.2. Train

In [32]:
train_sample_size_disaster_informative = 80_000
train_index_disaster_informative, gpu_index_disaster_informative = similarity.train_faiss_index(
    df_disaster_informative_embeddings,
    train_sample_size=train_sample_size_disaster_informative,
    nprobe=nprobe,
)

Building FAISS Index on CPU...
Transferring index to GPU for training...
Training index on 80000 samples...


In [9]:
# train_index_weather, gpu_index_weather = similarity.train_faiss_index(
#     df_weather_embeddings,
#     train_sample_size=train_sample_size_weather,
#     nprobe=nprobe,
# )

### C.3. Search

In [ ]:
# df_disaster_informative_drop_indices_radius = similarity.faiss_range_search(
#     df_disaster_informative_embeddings,
#     train_index_disaster_informative,
#     gpu_index=gpu_index_disaster_informative,
#     chunk_size=chunk_size,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_disaster_informative_drop_indices_radius.checkpoint.json',
# )

In [10]:
# df_weather_drop_indeices_radius = similarity.faiss_range_search(
#     df_weather_embeddings,
#     train_index_weather,
#     gpu_index=gpu_index_weather,
#     chunk_size=chunk_size,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_weather_drop_indeices_radius.checkpoint.json',
# )

In [11]:
# df_weather.drop(index=df_weather_drop_indeices_radius).reset_index(drop=True).to_csv(
#     extended_filepath / f"df_weather_radius_{similarity_threshold}.csv", index=False, quoting=csv.QUOTE_ALL)

In [ ]:
for similarity_threshold in [0.6, 0.65, 0.7, 0.75, 0.8]:
    df_disaster_informative_drop_indices_knearest = similarity.faiss_neighbors_search(
        df_disaster_informative_embeddings,
        train_index_disaster_informative,
        gpu_index=gpu_index_disaster_informative,
        chunk_size=chunk_size,
        top_k=top_k,
        similarity_threshold=similarity_threshold,
        checkpoint_file='df_disaster_informative_drop_indices_knearest.checkpoint.json',
    )
    df_disaster_informative.drop(
        index=df_disaster_informative_drop_indices_knearest
    ).reset_index(drop=True).to_csv(
        extended_filepath / f"df_disaster_informative_knearest_{similarity_threshold}_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)

Starting new FAISS knearest search...
Processed up to row 50000/244403. Identified 141538 near-duplicates.
Processed up to row 100000/244403. Identified 151687 near-duplicates.
Processed up to row 150000/244403. Identified 155414 near-duplicates.
Processed up to row 200000/244403. Identified 156857 near-duplicates.
Processed up to row 244403/244403. Identified 157140 near-duplicates.
Remove checkpoint file upon successful completion.
Starting new FAISS knearest search...
Processed up to row 50000/244403. Identified 119530 near-duplicates.
Processed up to row 100000/244403. Identified 129667 near-duplicates.
Processed up to row 150000/244403. Identified 133629 near-duplicates.
Processed up to row 200000/244403. Identified 135193 near-duplicates.
Processed up to row 244403/244403. Identified 135501 near-duplicates.
Remove checkpoint file upon successful completion.
Starting new FAISS knearest search...
Processed up to row 50000/244403. Identified 95061 near-duplicates.
Processed up to ro

In [ ]:
# df_weather_drop_indices_knearest = similarity.faiss_neighbors_search(
#     df_weather_embeddings,
#     train_index_weather,
#     gpu_index=gpu_index_weather,
#     chunk_size=chunk_size,
#     top_k=top_k,
#     similarity_threshold=similarity_threshold,
#     checkpoint_file='df_weather_knearest.checkpoint.json',
# )

In [ ]:
# df_weather.drop(index=df_weather_drop_indices_knearest).reset_index(drop=True).to_csv(
#     extended_filepath / f"df_weather_knearest_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)

In [5]:
# rule of thumb for nlist: 4 * sqrt(n) where n is the number of data points
n_out_topic = len(df_out_topic)
nlist_out_topic = int(4 * np.sqrt(n_out_topic))
train_sample_size_out_topic = int(n_out_topic / 10)

In [10]:
# df_out_topic_embeddings = similarity.vectorize_faiss(
#     df_out_topic, text_column='tweet_text', model_name=embedding_model, device='cuda'
# )
# np.savez(
#     extended_filepath / "df_out_topic_embeddings.npz",
#     embeddings=df_out_topic_embeddings,
#     model_name=embedding_model,
# )

df_out_topic_embeddings = np.load(
    extended_filepath / "df_out_topic_embeddings.npz",
    allow_pickle=True
)['embeddings']

In [ ]:
len(df_out_topic_embeddings)

10403525

In [11]:
train_index_out_topic, gpu_index_out_topic = similarity.train_faiss_index(
    df_out_topic_embeddings,
    train_sample_size=train_sample_size_out_topic,
    nprobe=nprobe,
)

Building FAISS Index on CPU...
Transferring index to GPU for training...


Training index on 1040352 samples...


In [ ]:
df_out_topic_drop_indeices_radius = similarity.faiss_range_search(
    df_out_topic_embeddings,
    train_index_out_topic,
    gpu_index=gpu_index_out_topic,
    chunk_size=chunk_size,
    similarity_threshold=similarity_threshold,
    checkpoint_file='df_out_topic_drop_indeices_radius.checkpoint.json',
)

Converting index back to CPU for range search...
Starting new FAISS radius search...
Processed up to row 50000/10403525. Identified 127328 near-duplicates.
Processed up to row 100000/10403525. Identified 152433 near-duplicates.
Processed up to row 150000/10403525. Identified 162242 near-duplicates.
Processed up to row 200000/10403525. Identified 168201 near-duplicates.
Processed up to row 250000/10403525. Identified 171547 near-duplicates.
Processed up to row 300000/10403525. Identified 173912 near-duplicates.
Processed up to row 350000/10403525. Identified 200018 near-duplicates.
Processed up to row 400000/10403525. Identified 208745 near-duplicates.
Processed up to row 450000/10403525. Identified 215555 near-duplicates.
Processed up to row 500000/10403525. Identified 250421 near-duplicates.
Processed up to row 550000/10403525. Identified 359521 near-duplicates.
Processed up to row 600000/10403525. Identified 416517 near-duplicates.
Processed up to row 650000/10403525. Identified 4571

In [ ]:
df_out_topic.drop(index=df_out_topic_drop_indeices_radius).reset_index(drop=True).to_csv(
    extended_filepath / f"df_out_topic_radius_{similarity_threshold}.csv", index=False, quoting=csv.QUOTE_ALL)

In [25]:
df_out_topic_knearest = similarity.faiss_neighbors_search(
    df_out_topic_embeddings,
    train_index_out_topic,
    gpu_index=gpu_index_out_topic,
    chunk_size=chunk_size,
    top_k=top_k,
    similarity_threshold=similarity_threshold,
    checkpoint_file='df_out_topic_knearest.checkpoint.json',
)

Starting new FAISS knearest search...


Processed up to row 50000/10403525. Identified 641301 near-duplicates.
Processed up to row 100000/10403525. Identified 750413 near-duplicates.
Processed up to row 150000/10403525. Identified 813423 near-duplicates.
Processed up to row 200000/10403525. Identified 860329 near-duplicates.
Processed up to row 250000/10403525. Identified 897449 near-duplicates.
Processed up to row 300000/10403525. Identified 931794 near-duplicates.
Processed up to row 350000/10403525. Identified 1043660 near-duplicates.
Processed up to row 400000/10403525. Identified 1060344 near-duplicates.
Processed up to row 450000/10403525. Identified 1070370 near-duplicates.
Processed up to row 500000/10403525. Identified 1351016 near-duplicates.
Processed up to row 550000/10403525. Identified 1994719 near-duplicates.
Processed up to row 600000/10403525. Identified 2335500 near-duplicates.
Processed up to row 650000/10403525. Identified 2579417 near-duplicates.
Processed up to row 700000/10403525. Identified 2784952 ne

In [26]:
df_out_topic.drop(index=df_out_topic_knearest).reset_index(drop=True).to_csv(
    extended_filepath / f"df_out_topic_knearest_{similarity_threshold}_top_{top_k}.csv", index=False, quoting=csv.QUOTE_ALL)